# Extraction Validation

  **Purpose.** Confirm the historical extraction is sane *before* downstream notebooks use it.
  Asks five questions of every (zone, endpoint, month) file on disk:

  1. Does it exist where we expect?
  2. Does it have the expected shape and schema?
  3. Did any sandbox-mode data leak through?
  4. Where are the missing values, and are they realistic?
  5. Do the values look right when plotted?

  **Scope.** Read-only. This notebook never writes to `data/`. If anything looks wrong, fix the
  cause upstream (extraction code, EM endpoint, weather coordinate) and re-run the relevant
  months.

In [ ]:
"""
What this does:
  1. from __future__ import annotations — same modern-typing convention as the rest of the codebase, so any type hints we write later work without imports.
  2. pio.templates.default = "plotly_white" applies the locked plot style globally so we don't repeat it.
  3. DATA_ROOT = Path("../data") notebook lives one level down (notebooks/), so .. walks back to the project root.
  4. The print at the end is a sanity check that the kernel resolved imports against the right .venv and that ZONES matches expectations.
"""

In [ ]:
# Libraries and imports
from __future__ import annotations
from pathlib import Path

import pandas as pd
import plotly.express as px

from carbon_forecast.data.storage import read_parquet
from carbon_forecast.data.zones import ZONES
from carbon_forecast.plotting.config import apply_defaults

import carbon_forecast

# Set project root path and data root paths
PROJECT_ROOT = Path(carbon_forecast.__file__).resolve().parents[2]
DATA_ROOT = PROJECT_ROOT / "data"

# Apply the locked plotly defaults (template + font, etc.).
apply_defaults()

# State the 3 historical endpoints for ci, power, flows
EM_ENDPOINTS = (
    "carbon-intensity/past-range",
    "power-breakdown/past-range",
    "electricity-flows/past-range",
)

# Print check
print(f"Project zones: {[z.em_key for z in ZONES]}")
print(f"EM endpoints: {list(EM_ENDPOINTS)}")
print(f"Data root: {DATA_ROOT.resolve()}")


In [ ]:
print("Resolved:", DATA_ROOT.resolve())
print("Exists  :", DATA_ROOT.exists())
print("Sample  :", list(DATA_ROOT.glob("raw/em/BE/*")))

## Inventory

  Walk `data/raw/` and build a tidy DataFrame of every (zone, source, endpoint, year, month)
  Parquet on disk. Two follow-up tables:

  - **Coverage matrix.** Count of months present per (zone, endpoint). Confirms each cell has the
  same depth as the others.
  - **Gap report.** Months missing from the requested 2021-01 → 2025-12 window. While extraction is
   running, this list shrinks; once extraction completes, it should be empty.

In [ ]:
import re

_MONTH_PATTERN = re.compile(r"^(\d{4})-(\d{2})\.parquet$")


def inventory_files(data_root: Path) -> pd.DataFrame:
    """
    One row per Parquet file found under data_root/raw/.

    What this does:
    - Walks data/raw/em/{zone}/{endpoint}/{group}/{YYYY-MM}.parquet.
      The 'group' extra level exists because EM endpoint URLs are two
      segments (e.g. 'carbon-intensity/past-range'): 'carbon-intensity'
      is the endpoint dir, 'past-range' is the dir below.
    - Walks data/raw/weather/{zone}/{YYYY-MM}.parquet.
    - Returns one row per Parquet with parsed year/month and the full path.
    """
    rows = []

    em_root = data_root / "raw/em"
    if em_root.exists():
        for zone_dir in sorted(p for p in em_root.iterdir() if p.is_dir() and p.name != "forecasts"):
            for endpoint_dir in sorted(p for p in zone_dir.iterdir() if p.is_dir()):
                for f in sorted(endpoint_dir.glob("*/*.parquet")):
                    m = _MONTH_PATTERN.match(f.name)
                    if not m:
                        continue
                    year, month = int(m.group(1)), int(m.group(2))
                    endpoint = f"{endpoint_dir.name}/{f.parent.name}"
                    rows.append((zone_dir.name, "em", endpoint, year, month, f))

    weather_root = data_root / "raw/weather"
    if weather_root.exists():
        for zone_dir in sorted(p for p in weather_root.iterdir() if p.is_dir()):
            for f in sorted(zone_dir.glob("*.parquet")):
                m = _MONTH_PATTERN.match(f.name)
                if not m:
                    continue
                year, month = int(m.group(1)), int(m.group(2))
                rows.append((zone_dir.name, "weather", "archive", year, month, f))

    return pd.DataFrame(rows, columns=["zone", "source", "endpoint", "year", "month", "path"])


inv = inventory_files(DATA_ROOT)
print(f"Files on disk: {len(inv)}")
inv.head()


In [ ]:
coverage = (
      inv.groupby(["zone", "source", "endpoint"])
      .size()
      .rename("months_present")
      .reset_index()
      .pivot_table(
          index=["zone"],
          columns=["source", "endpoint"],
          values="months_present",
          fill_value=0,
      )
  )
coverage

## Shape and schema

For each Parquet on disk, check four invariants and one schema:

- **Row count** matches the expected hours-in-month (`24 * days_in_month`). Slight under-counts are tolerable for in-progress months; under-counts inside a closed month flag missing hours.
- **Index** is a UTC `DatetimeIndex`, **monotonic increasing**, and **duplicate-free**.
- **Schema** matches the column pattern expected for that endpoint:
  - `carbon-intensity` -> exactly `{carbon_intensity_gco2eq_kwh}`.
  - `power-breakdown` -> columns prefixed `prod_*_mw` / `cons_*_mw`, plus optional `import_total_mw` / `export_total_mw`.
  - `electricity-flows` -> columns prefixed `import_*_mw` / `export_*_mw`.
  - `weather` -> the six locked Open-Meteo variables.

The output is a long DataFrame (`checks`); the cell at the end prints only the rows that failed at least one check.


In [ ]:
from calendar import monthrange
from datetime import datetime, timezone

from carbon_forecast.data.weather_client import HOURLY_VARIABLES

WEATHER_EXPECTED_COLS = set(HOURLY_VARIABLES)


def _schema_ok(endpoint: str, cols: set[str]) -> bool:
    if endpoint == "carbon-intensity/past-range":
        return cols == {"carbon_intensity_gco2eq_kwh"}
    if endpoint == "power-breakdown/past-range":
        prefixed = all(
            c.startswith(("prod_", "cons_")) or c in {"import_total_mw", "export_total_mw"}
            for c in cols
        )
        return prefixed and len(cols) > 0
    if endpoint == "electricity-flows/past-range":
        return all(c.startswith(("import_", "export_")) for c in cols) and len(cols) > 0
    if endpoint == "archive":
        return cols == WEATHER_EXPECTED_COLS
    return False


def _check_one(row: pd.Series) -> dict:
    df = read_parquet(row["path"])
    days = monthrange(row["year"], row["month"])[1]
    expected_rows = 24 * days
    cols = set(df.columns)
    return {
        "rows": len(df),
        "expected_rows": expected_rows,
        "row_count_ok": len(df) == expected_rows,
        "index_utc": str(df.index.tz) == "UTC",
        "index_monotonic": df.index.is_monotonic_increasing,
        "index_unique": df.index.is_unique,
        "n_cols": len(cols),
        "schema_ok": _schema_ok(row["endpoint"], cols),
    }


checks = pd.concat([inv, inv.apply(_check_one, axis=1, result_type="expand")], axis=1)
checks["all_ok"] = checks[["row_count_ok", "index_utc", "index_monotonic", "index_unique", "schema_ok"]].all(axis=1)

print(f"Checked {len(checks)} files; {checks['all_ok'].sum()} pass, {(~checks['all_ok']).sum()} fail.")
checks.head()


## Physical sanity

  The client raises `SandboxModeError` before flattening, so sandbox data cannot reach disk by
  construction. This section verifies the post-condition: each numeric column lies inside a hard
  physical range. Anything outside the range is either a sandbox slip, a unit error, or a real
  outlier worth investigating.

  Ranges per column family:

  - Carbon intensity (`carbon_intensity_gco2eq_kwh`): `[0, 2000]` gCO2eq/kWh. Real grids span ~5
  (Norwegian hydro) to ~1100 (Polish coal); 2000 is a hard ceiling.
  - Power-breakdown (`prod_*_mw`, `cons_*_mw`, `import_total_mw`, `export_total_mw`): `[-50000,
  200000]` MW. Negative values are valid for storage discharge sign conventions; magnitudes above a
  few hundred GW are impossible at zone scale.
  - Electricity-flows (`import_*_mw`, `export_*_mw`): `[0, 50000]` MW. Directional, so non-negative.
  - Weather temperature (`temperature_2m`, `dewpoint_2m`): `[-60, 60]` °C.
  - Weather wind speed (`wind_speed_10m`): `[0, 100]` m/s.
  - Weather wind direction (`wind_direction_10m`): `[0, 360]` degrees.
  - Weather radiation (`shortwave_radiation`): `[0, 1500]` W/m².
  - Weather precipitation (`precipitation`): `[0, 200]` mm/h.

In [ ]:
from ipywidgets.widgets import widget_upload
from ipywidgets.widgets import widget_upload
from ipywidgets.widgets import widget_upload
from ipywidgets.widgets import widget_upload
import numpy as np

# (lower, upper) hard physical bounds per column or column pattern.
RANGE_EXACT = {
    "carbon_intensity_gco2eq_kwh": (0.0, 2000.0),
    "import_total_mw": (-1000.0, 200_000.0),
    "export_total_mw": (-1000.0, 200_000.0),
    "temperature_2m": (-60.0, 60.0),
    "dewpoint_2m": (-60.0, 60.0),
    "wind_speed_10m": (0.0, 100.0),
    "wind_direction_10m": (0.0, 360.0),
    "shortwave_radiation": (0.0, 1500.0),
    "precipitation": (0.0, 200.0),
}
RANGE_BY_PREFIX = {
    "prod_": (-50_000.0, 200_000.0),
    "cons_": (-50_000.0, 200_000.0),
    "import_": (0.0, 50_000.0),
    "export_": (0.0, 50_000.0),
}


def _bounds_for(col: str) -> tuple[float, float] | None:
    if col in RANGE_EXACT:
        return RANGE_EXACT[col]
    for prefix, bounds in RANGE_BY_PREFIX.items():
        if col.startswith(prefix):
            return bounds
    return None


def _value_check(row: pd.Series) -> dict:
    df = read_parquet(row["path"])
    out_of_range = 0
    constant_nonzero = 0
    n_numeric = 0
    offending: list[str] = []

    for col in df.columns:
        s = df[col].dropna()
        if s.empty or not np.issubdtype(s.dtype, np.number):
            continue
        n_numeric += 1
        bounds = _bounds_for(col)
        if bounds is not None and ((s.min() < bounds[0]) or (s.max() > bounds[1])):
            out_of_range += 1
            offending.append(f"{col} [{s.min():.1f}, {s.max():.1f}]")
        # Constant *non-zero* is the real sandbox / stuck-sensor smell.
        # Constant zero just means the zone lacks that fuel or that interconnector
        # is sub-reporting-threshold in this month -- physically valid.
        if s.nunique() == 1 and s.iloc[0] != 0 and len(s) > 24:
            constant_nonzero += 1
            offending.append(f"{col}=const({s.iloc[0]})")

    return {
        "n_numeric_cols": n_numeric,
        "out_of_range_cols": out_of_range,
        "constant_nonzero_cols": constant_nonzero,
        "value_check_ok": (out_of_range == 0) and (constant_nonzero == 0),
        "offending": "; ".join(offending) if offending else "",
    }


value_checks = pd.concat(
    [inv, inv.apply(_value_check, axis=1, result_type="expand")], axis=1
)

print(f"Files audited: {len(value_checks)}; "
    f"clean: {value_checks['value_check_ok'].sum()}; "
    f"with issues: {(~value_checks['value_check_ok']).sum()}")
value_checks[~value_checks["value_check_ok"]].head(20)

## Missing values

  NaN policy varies by column family:

  - **Carbon intensity** is a single number per hour. NaN ratio should be near zero; anything over
  5% is a data gap.
  - **Power breakdown** (`prod_*`, `cons_*`) can be moderately sparse — a zone's fuel mix changes
  hour to hour and some sources are seasonal. Threshold: 30% NaN per column is the soft alarm.
  - **Electricity flows** (`import_*`, `export_*`) are mutually exclusive per direction per hour, so
   40-60% NaN per column is the *normal* shape. We only flag a column that is **100% NaN** for a
  month, which means that interconnector reported nothing all month.
  - **Weather** variables come from a continuous reanalysis; NaN ratio over 1% suggests a real fetch
   problem.

  Output: one row per file with NaN ratios summarised, plus a printout of any rows tripping a
  threshold.

In [ ]:
def _nan_check(row: pd.Series) -> dict:
    df = read_parquet(row["path"])
    endpoint = row["endpoint"]
    n = len(df) if len(df) else 1  # guard against zero-row files

    nan_ratios = df.isna().sum() / n
    flagged: list[str] = []

    if endpoint == "carbon-intensity/past-range":
        bad = nan_ratios[nan_ratios > 0.05]
        flagged.extend(f"{c}={r:.2%}" for c, r in bad.items())
    elif endpoint == "power-breakdown/past-range":
        breakdown_cols = nan_ratios.filter(regex=r"^(prod|cons)_")
        bad = breakdown_cols[breakdown_cols > 0.30]
        flagged.extend(f"{c}={r:.2%}" for c, r in bad.items())
    elif endpoint == "electricity-flows/past-range":
        bad = nan_ratios[nan_ratios >= 1.0]
        flagged.extend(f"{c}=100%" for c, r in bad.items())
    elif endpoint == "archive":
        bad = nan_ratios[nan_ratios > 0.01]
        flagged.extend(f"{c}={r:.2%}" for c, r in bad.items())

    return {
        "max_nan_ratio": float(nan_ratios.max()) if not nan_ratios.empty else 0.0,
        "mean_nan_ratio": float(nan_ratios.mean()) if not nan_ratios.empty else 0.0,
        "nan_check_ok": len(flagged) == 0,
        "nan_offending": "; ".join(flagged),
    }


nan_checks = pd.concat([inv, inv.apply(_nan_check, axis=1, result_type="expand")], axis=1)

print(f"Files audited: {len(nan_checks)}; "
    f"clean: {nan_checks['nan_check_ok'].sum()}; "
    f"flagged: {(~nan_checks['nan_check_ok']).sum()}")
nan_checks[~nan_checks["nan_check_ok"]].sort_values("max_nan_ratio", ascending=False).head(20)

## Sample plots

Four spot checks against plausibility:

1. **Carbon intensity, June 2023, all five zones overlaid.** Confirms the 5-year, 5-zone time series load and that zone ordering matches expectations.
2. **Production mix stacked area, BE January 2024.** Confirms `prod_*` columns add up to a realistic ~10 GW peak and the fuel mix matches Belgium's known mix.
3. **Net cross-border flows, BE January 2024.** Confirms `import_*` / `export_*` columns are non-negative and partner zones look right (FR, DE, NL, GB, LU).
4. **Temperature and solar irradiance, BE January 2024.** Confirms weather is winter-shaped: cold temperatures, low solar irradiance.

Palettes and the layout helper live in `src/carbon_forecast/plotting/config.py`; we import them once below.


In [ ]:
from carbon_forecast.plotting.config import (
    ENERGY_PALETTE,
    ENERGY_SOURCE_ORDER,
    PLOT_H,
    PLOT_W,
    REGIONAL_PALETTE,
    style_fig,
)


In [ ]:
def load_zone_month(endpoint: str, year: int, month: int) -> pd.DataFrame:
    """Concat one (endpoint, year, month) slice across all zones present on disk."""
    rows = inv.query(
        "endpoint == @endpoint and year == @year and month == @month"
    )
    frames = []
    for _, r in rows.iterrows():
        df = read_parquet(r["path"]).reset_index()
        df["zone"] = r["zone"]
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


ci_jun2023 = load_zone_month("carbon-intensity/past-range", 2023, 6)
print(f"CI rows loaded: {len(ci_jun2023)}; zones: {sorted(ci_jun2023['zone'].unique())}")

fig = px.line(
    ci_jun2023,
    x="datetime",
    y="carbon_intensity_gco2eq_kwh",
    color="zone",
    color_discrete_map=REGIONAL_PALETTE,
    category_orders={"zone": list(REGIONAL_PALETTE.keys())},
    labels={
        "datetime": "Hour (UTC)",
        "carbon_intensity_gco2eq_kwh": "Carbon intensity (gCO2eq/kWh)",
        "zone": "Zone",
    },
)
style_fig(fig, "Carbon intensity in June 2023")
fig.show()


In [ ]:
be_pb_2024_01 = read_parquet(DATA_ROOT / "raw/em/BE/power-breakdown/past-range/2024-01.parquet")

# Keep only prod_*_mw columns; strip prefix/suffix to get the source name.
prod_cols = [c for c in be_pb_2024_01.columns if c.startswith("prod_") and c.endswith("_mw")]
prod_df = be_pb_2024_01[prod_cols].copy()
prod_df.columns = [c[len("prod_"):-len("_mw")] for c in prod_cols]

# Wide -> long for plotly.
prod_long = (
    prod_df.reset_index()
    .melt(id_vars="datetime", var_name="source", value_name="mw")
    .dropna(subset=["mw"])
)

# Canonical stacking order from plotting.config: baseload (nuclear) at bottom,
# fossils on top. Filter to sources actually present in this month.
present = [s for s in ENERGY_SOURCE_ORDER if s in prod_long["source"].unique()]

fig = px.area(
    prod_long,
    x="datetime",
    y="mw",
    color="source",
    color_discrete_map=ENERGY_PALETTE,
    category_orders={"source": present},
    labels={"datetime": "UTC time", "mw": "Production (MW)", "source": "Source"},
)
style_fig(fig, "Belgium Production Mix in January 2024 (hourly, MW)")
fig.show()


In [ ]:
be_flows_2024_01 = read_parquet(
    DATA_ROOT / "raw/em/BE/electricity-flows/past-range/2024-01.parquet"
)

# Collect partner zones present in this file by stripping prefixes/suffixes.
import_cols = [c for c in be_flows_2024_01.columns if c.startswith("import_") and c.endswith("_mw")]
export_cols = [c for c in be_flows_2024_01.columns if c.startswith("export_") and c.endswith("_mw")]
partners = sorted({c[len("import_"):-len("_mw")] for c in import_cols}
                  | {c[len("export_"):-len("_mw")] for c in export_cols})

# Net flow per partner: positive = BE importing, negative = BE exporting.
net = pd.DataFrame(index=be_flows_2024_01.index)
for p in partners:
    imp = be_flows_2024_01.get(f"import_{p}_mw", pd.Series(0.0, index=be_flows_2024_01.index)).fillna(0.0)
    exp = be_flows_2024_01.get(f"export_{p}_mw", pd.Series(0.0, index=be_flows_2024_01.index)).fillna(0.0)
    net[p.upper()] = imp - exp

net_long = net.reset_index().melt(id_vars="datetime", var_name="partner", value_name="net_mw")

fig = px.line(
    net_long,
    x="datetime",
    y="net_mw",
    color="partner",
    color_discrete_sequence=px.colors.qualitative.Bold,
    labels={"datetime": "UTC time", "net_mw": "Net flow (MW, +import / -export)", "partner": "Partner"},
)
fig.add_hline(y=0, line_dash="dot", line_color="rgba(0,0,0,0.4)")
style_fig(fig, "Belgium Net Cross-border Flows in January 2024")
fig.show()


In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

be_wx_2024_01 = read_parquet(DATA_ROOT / "raw/weather/BE/2024-01.parquet")

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=be_wx_2024_01.index,
        y=be_wx_2024_01["temperature_2m"],
        name="Temperature (°C)",
        line=dict(color="#D55E00", width=1.5),
    ),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=be_wx_2024_01.index,
        y=be_wx_2024_01["shortwave_radiation"],
        name="Shortwave radiation (W/m²)",
        line=dict(color="#F4D03F", width=1.5),
        opacity=0.85,
    ),
    secondary_y=True,
)

fig.update_xaxes(title_text="UTC time")
fig.update_yaxes(title_text="Temperature (°C)", secondary_y=False)
fig.update_yaxes(title_text="Shortwave radiation (W/m²)", secondary_y=True)
style_fig(fig, "Belgium weather in January 2024")
fig.show()


## Findings

The extraction pipeline produced clean, consistent data across all five zones and the full 2021-01
to 2025-12 window. Summary:

**Inventory (Section 2).** 1,200 Parquet files on disk: 5 zones × 60 months × 3 EM endpoints (900)
plus 5 zones × 60 months of weather (300). Matches the full target; no missing months.

**Shape and schema (Section 3).** All 1,200 files pass the row-count, UTC-index, monotonic,
unique-index, and schema checks.

**Physical sanity (Section 4).** All 1,200 files pass the bounded-range and non-zero-constant
checks. The initial run flagged 294 files for zero-constant columns; on inspection these were
legitimately-zero columns (fuels the zone does not have: SG has no nuclear, wind, or hydro; BE has
no geothermal; battery storage barely exists in the 2021-2023 window). The check was relaxed to
flag only non-zero constants, which is the real sandbox or stuck-sensor signal.

**Missing values (Section 5).** Six files tripped the per-family NaN thresholds. All six are
physically explainable and accepted as known sparsity:

- SG 2024-03 and 2024-04 `power-breakdown`: `prod_nuclear_mw`, `prod_wind_mw`, `prod_hydro_mw` at
66 to 99% NaN. Singapore lacks these fuels; EM's reporting convention shifted mid-2024 from
explicit zero to NaN.
- BE 2024-12 `power-breakdown`: `prod_battery_discharge_mw` at 99.87% NaN. Belgium has minimal
grid-scale storage; discharge is reported as NaN when no storage is dispatched.
- FI 2022-01, 2022-09, 2022-12 `power-breakdown`: `prod_solar_mw` at 99% NaN. Finland sits at
60°N+; mid-winter solar production is effectively zero and EM reports the zero hours as NaN.

**Visual sanity (Section 6).** Four cross-checks held up:

- Cross-region carbon intensity in June 2023 shows the expected ranking: FI (low, nuclear plus
hydro) below BE (mixed, highest amplitude swings), with both well below SG (gas-dominated, narrow
band around 500), and the two US zones in the middle. BE shows the largest amplitude variance
among the five, consistent with its high-mix portfolio swinging between near-zero-CI nuclear/wind
and high-CI gas as the marginal generator changes hour to hour. This is a finding worth carrying
into the per-region heterogeneity atlas (Contribution 3).
- BE January 2024 production mix shows the expected ~12 GW winter peak, nuclear baseload around 4
to 5 GW, variable wind, weak solar, and gas filling the gap during low-wind hours.
- BE January 2024 cross-border flows show net export to FR most hours and net import from DE
during German high-wind periods, with smaller flows to NL, LU, GB, all peaking around 1 to 2 GW
per partner.
- BE January 2024 weather is winter-shaped: temperatures between -5 and +12 °C with daily cycles,
shortwave radiation zero at night and peaking 200 to 400 W/m² midday.